## Creating Custom MCP Tools

# Lesson 7: Native Extensions and Custom Tool Architectures via In-Process MCP

In the previous module, you discovered how to scale your runtime environment using external **Model Context Protocol (MCP)** servers, connecting pre-built tool ecosystems like documentation indexers and remote databases.

While community packages address generic engineering operations, software platforms often require specialized utility hooks deeply coupled with your application's unique state, internal APIs, or tracking states.

In this lesson, you will master how to expose raw Python logic directly to Claude as an in-process, protocol-compliant MCP tool suite using the `@tool` decorator layer.

---

## The Structural Anatomy of a Custom Tool Function

For the underlying Claude Code subprocess engine to execute a custom Python utility, your target function must follow a specific signature interface. Every programmatic tool maps directly to an asynchronous function accepting a single parameter payload and yielding a structured text envelope.

```python
from typing import Any

async def my_custom_tool(args: dict[str, Any]) -> dict[str, Any]:
    # 1. Extract inputs from the standardized args envelope
    param_value = args["parameter_key"]
    
    # 2. Execute localized backend or system core computations
    computation_output = f"Processed target value: {param_value}"
    
    # 3. Encapsulate response text inside the strict protocol matrix
    return {
        "content": [
            {
                "type": "text", 
                "text": computation_output
            }
        ]
    }

```

### Protocol Structural Invariants:

* **Asynchronous Signatures (`async def`):** The engine executes tools concurrently within a non-blocking asynchronous network pipeline. Standard blocking declarations (`def`) are strictly invalid.
* **The `args` Ingestion Map:** Rather than utilizing standard Python positional or keyword arguments, the tool takes exactly one object: a dictionary (`dict[str, Any]`) holding all variables injected by the model.
* **The `"content"` Envelope Layout:** The function return type must resolve to a dictionary holding a `"content"` list key. Even if your tool yields a single primitive string response, that string must sit wrapped within a content block dictionary mapping `"type": "text"` and `"text": "<data>"`.

---

## Defining Metadata via the `@tool` Decorator

Once you lay out a function matching the structural syntax rules, decorate it with the `@tool` configuration boundary. This decorator acts as the semantic schema registry—compiling your code signature into JSON metadata that Claude parses to discover when and how to invoke the logic.

```python
from claude_agent_sdk import tool

@tool(
    name="calculate_hash",
    description="Generates an internal system tracking hash payload. Use this when finalizing commits.",
    input_schema={"data_payload": str}
)
async def calculate_hash(args: dict[str, Any]) -> dict[str, Any]:
    raw_data = args["data_payload"]
    # Internal tracking logic goes here...
    return {"content": [{"type": "text", "text": "HASH_ID_7781"}]}

```

### Parameter Schema Specification:

* **`name`:** The string identifier presented directly to the model. Use snake_case formatting for tool tokens.
* **`description`:** The semantic operational manual read by the model during its reasoning chain. The text must be specific, explaining exactly what side effects the function triggers and when it is appropriate to use.
* **`input_schema`:** A type-mapping dictionary enforcing argument constraints. Defining `{"data_payload": str}` guarantees that when Claude calls the tool, the SDK verifies the input and seeds `args["data_payload"]` as a clean string.

---

## Bundling and Whitelisting In-Process Servers

After building your tools, bundle them into a unified in-process protocol layer using the `create_sdk_mcp_server()` method, and map them to your workspace options:

```python
from pathlib import Path
from claude_agent_sdk import create_sdk_mcp_server, ClaudeAgentOptions

# 1. Bundle isolated tool hooks into a single server container block
custom_utility_server = create_sdk_mcp_server(
    name="internal_utils",
    version="1.0.0",
    tools=[calculate_hash] # Registers your decorated function
)

# 2. Map the localized server node directly into your runtime options
options = ClaudeAgentOptions(
    model="haiku",
    max_turns=10,
    mcp_servers={
        "system_hooks": custom_utility_server # Binds server to unique key
    },
    allowed_tools=[
        # Whitelist format: mcp__<server_key>__<tool_name>
        "mcp__system_hooks__calculate_hash"
    ],
    cwd=Path(__file__).parent
)

```

The system treats in-process servers identically to external processes. The unique mapping key you select inside `mcp_servers` (`"system_hooks"`) binds directly to the namespaced prefix constraints required in your `allowed_tools` allowlist array: `mcp__system_hooks__calculate_hash`.

---

## Case Study: Implementing a Stateful Multi-Turn Planner Suite

To see how custom tools can fundamentally change an agent's problem-solving behavior, let's build a **Stateful Task Planner System**.

By introducing custom tools that explicitly read and write to an tracking state array, we can force Claude to act as a highly methodical engineer—breaking complex repository rearrangements down into a visible checklist before executing any dangerous shell commands.

```python
import anyio
from typing import Any
from pathlib import Path
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, tool, create_sdk_mcp_server
from utils import display_response

# =====================================================================
# GLOBAL SESSION MEMORY STATE
# =====================================================================
PLAN: list[dict[str, str]] = []

# =====================================================================
# TOOL 1: APPENDING STEPS TO STATE
# =====================================================================
@tool(
    name="add_to_plan",
    description="Add one or more sequential steps to your operational checklist. Use this before executing commands.",
    input_schema={"steps": list}
)
async def add_to_plan(args: dict[str, Any]) -> dict[str, Any]:
    target_steps = args["steps"]
    for step in target_steps:
        PLAN.append({"task": str(step), "status": "pending"})
    return {"content": [{"type": "text", "text": f"Successfully registered {len(target_steps)} tasks to master plan."}]}

# =====================================================================
# TOOL 2: INSPECTING RECOVERY MATRIX
# =====================================================================
@tool(
    name="read_plan",
    description="Inspect the current tracking checklist state to view pending and completed steps.",
    input_schema={}
)
async def read_plan(args: dict[str, Any]) -> dict[str, Any]:
    if not PLAN:
        return {"content": [{"type": "text", "text": "Master plan checklist is empty."}]}
    
    view_stream = "Master Operational Plan:\n"
    for index, record in enumerate(PLAN):
        status_flag = "✅ Done" if record["status"] == "done" else "⬜ Pending"
        view_stream += f"{index + 1}. [{status_flag}] -> {record['task']}\n"
    return {"content": [{"type": "text", "text": view_stream}]}

# =====================================================================
# TOOL 3: TRANSITIONING STATUS KEYS
# =====================================================================
@tool(
    name="mark_step_done",
    description="Transition a specific plan index step to complete (uses 1-based human indexing).",
    input_schema={"step_index": int}
)
async def mark_step_done(args: dict[str, Any]) -> dict[str, Any]:
    target_index = args["step_index"] - 1
    
    if 0 <= target_index < len(PLAN):
        PLAN[target_index]["status"] = "done"
        return {"content": [{"type": "text", "text": f"Step #{args['step_index']} marked complete."}]}
    
    # Return a structured protocol error frame if index is out-of-bounds
    return {
        "content": [{"type": "text", "text": f"Execution Error: Index {args['step_index']} out of range."}],
        "isError": True
    }

```

---

## Orchestrating the Methodical Agent Pipeline

Now, bundle these three interconnected tools into a `"planner"` server context, attach core built-in terminal tools (`Bash`, `Read`), and enforce the planning behavior via a structured system prompt:

```python
async def main():
    # 1. Compile stateful planner server node
    planner_server = create_sdk_mcp_server(
        name="task_planner",
        version="1.0.0",
        tools=[add_to_plan, read_plan, mark_step_done]
    )
    
    # 2. Package hybrid permissions options matrix
    options = ClaudeAgentOptions(
        model="haiku",
        max_turns=30, # High threshold to accommodate the planning overhead
        mcp_servers={
            "planning": planner_server
        },
        allowed_tools=[
            "mcp__planning__add_to_plan",
            "mcp__planning__read_plan",
            "mcp__planning__mark_step_done",
            "Bash",
            "Read"
        ],
        permission_mode="acceptEdits",
        cwd=str(Path(__file__).parent / "project"),
        # Enforce step-by-step verification rules via system instruction
        system_prompt=(
            "You are a methodical software release agent. For any complex file reorganization, "
            "you MUST follow this strict structural protocol:\n"
            "1. Run 'add_to_plan' to outline your steps before executing shell changes.\n"
            "2. Run tasks sequentially.\n"
            "3. After completing a task, call 'mark_step_done' to verify the step.\n"
            "Keep the tracking matrix current at all times."
        )
    )
    
    # 3. Launch execution context against a unorganized workspace folder
    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            "This directory contains unorganized flat layout files (config.json, script.js, style.css). "
            "Analyze the workspace contents, establish an architectural folder schema plan, and "
            "execute a clean reorganization moving files into logical subdirectories."
        )
        await display_response(client)

if __name__ == "__main__":
    anyio.run(main)

```

---

## Step-by-Step Diagnostic Trace

When running this program, watch the terminal logs to see how the agent orchestrates the workflow. Notice how it seamlessly alternates between custom state tracking tools and standard built-in terminal operations:

### Turn 1: Inspection & Analysis

The agent scans the messy directory using `Bash` and checks the file extensions using `Read` to map out its plan:

```text
💬 Claude Response:
I am scanning the root project files to build an understanding of the extension types before designing our layout folders.

🔧 [Tool Invocation]: Bash
  -> command: ls -la

🔧 [Tool Invocation]: Read
  -> file_path: config.json

```

### Turn 2: Initializing the Structural Plan

With its folder strategy set, the agent initializes the custom state tracking manager before running any modifying system code:

```text
💬 Claude Response:
I have mapped the project state. Before modifying files, I will register a multi-step checklist inside our master planner server.

🔧 [Tool Invocation]: mcp__planning__add_to_plan
  -> steps: ['Create directories config/, src/, assets/', 'Move config.json to config/', 'Move script.js to src/', 'Verify new structure']

✅ [Tool Output Block Received]
  -> Data Trace: Successfully registered 4 tasks to master plan.

```

### Turn 3: Executing Steps and Incrementing State

The agent creates the directories via a built-in tool, then immediately transitions the tracking index using your custom module state logic:

```text
🔧 [Tool Invocation]: Bash
  -> command: mkdir -p config src assets

🔧 [Tool Invocation]: mcp__planning__mark_step_done
  -> step_index: 1

✅ [Tool Output Block Received]
  -> Data Trace: Step #1 marked complete.

```

### Turn 4: Sequential File Relocation

Claude moves the target script configurations down the folder tree and increments the matching task flags:

```text
🔧 [Tool Invocation]: Bash
  -> command: mv config.json config/ && mv script.js src/

🔧 [Tool Invocation]: mcp__planning__mark_step_done
  -> step_index: 2

```

### Turn 5: Final Inspection and Verification Summary

The agent verifies the directory structure is pristine, marks the final tracking node complete, and outputs a clean technical summary block:

```text
🔧 [Tool Invocation]: Bash
  -> command: ls -R

🔧 [Tool Invocation]: mcp__planning__mark_step_done
  -> step_index: 4

💬 Claude Response:
## Reorganization Complete

The project layout has been restructured cleanly:
project/
├── assets/
│   └── style.css
├── config/
│   └── config.json
└── src/
    └── script.js

All steps have been executed, tracked, and verified through the internal master plan checklist!

```

---

## Summary

You have mastered how to build native in-process extensions using the Model Context Protocol:

* **The `async def` and `args` schemas** provide a standardized, protocol-compliant layout for injecting custom logic into the runtime pipeline.
* **The `@tool` decorator parameters** compile your Python documentation and argument validation rules into clear JSON schemas for the model.
* **`create_sdk_mcp_server()`** packages multiple native functions into a single named, versioned server node.
* Integrating custom tools with state structures creates highly sophisticated agents that can plan, verify, and track complex work loops with complete consistency.

## Testing Planning Tools Manually

Now that you've learned how to create custom MCP tools with the @tool decorator, it's time to understand the underlying structure by working with the planning functions directly before they become tools.

In this exercise, you'll manually test three planning functions (add_to_plan, read_plan, and mark_step_done) to understand the data structures they expect and the results they return. These are plain Python functions, your job is to call them in the main function and print their results.

Follow the TODO comments in the main function to:

    Call add_to_plan with a dictionary containing a "steps" key and a list of 2–3 step strings.
    Print the result to see the confirmation message.
    Call read_plan with an empty dictionary to view the current plan.
    Print the result to see the formatted plan with pending steps.
    Call mark_step_done with a dictionary containing "index": 1 to mark the first step as complete.
    Print the result to see the confirmation.
    Call read_plan again and print the result to see the updated plan with one step marked as done.

By the end of this exercise, you'll have a clear understanding of how the planning functions work before you add the @tool decorator to turn them into MCP tools in future exercises!


```
from typing import Any

# --- Planning State ---
PLAN = []

# --- Planning Functions ---

def add_to_plan(args: dict[str, Any]) -> dict[str, Any]:
    steps = args["steps"]
    added_count = 0
    
    for step in steps:
        PLAN.append({"step": step, "status": "pending"})
        added_count += 1
        
    return {
        "content": [{"type": "text", "text": f"Added {added_count} steps to plan."}]
    }


def read_plan(args: dict[str, Any]) -> dict[str, Any]:
    if not PLAN:
        return {"content": [{"type": "text", "text": "Plan is empty."}]}
    
    result = "Current Plan:\n"
    for i, item in enumerate(PLAN):
        icon = "✅" if item["status"] == "done" else "⬜"
        result += f"{i+1}. {icon} {item['step']}\n"
        
    return {
        "content": [{"type": "text", "text": result}]
    }


def mark_step_done(args: dict[str, Any]) -> dict[str, Any]:
    idx = args["index"] - 1
    if 0 <= idx < len(PLAN):
        PLAN[idx]["status"] = "done"
        return {
            "content": [{"type": "text", "text": f"Marked step #{args['index']} as complete."}]
        }
    else:
        return {
            "content": [{"type": "text", "text": f"Error: Invalid step index {args['index']}."}],
            "isError": True
        }


def main():
    # TODO: Call add_to_plan with a dictionary containing a "steps" key with a list of 2-3 step strings (e.g., "Write code", "Test code", "Deploy code")
    
    # TODO: Print the result from add_to_plan
    
    # TODO: Call read_plan with an empty args dictionary {}
    
    # TODO: Print the result from read_plan
    
    # TODO: Call mark_step_done with a dictionary containing "index": 1
    
    # TODO: Print the result from mark_step_done
    
    # TODO: Call read_plan again to see the updated state
    
    # TODO: Print the final result


if __name__ == "__main__":
    main()
```

Here is the completed script with the synchronous python testing logic added directly to your `main()` function execution tree.

Running this file locally prints out a clear telemetry log showing exactly how these dictionary payloads pass inputs and return strict protocol-compliant maps:

```python
from typing import Any

# --- Planning State ---
PLAN = []

# --- Planning Functions ---

def add_to_plan(args: dict[str, Any]) -> dict[str, Any]:
    steps = args["steps"]
    added_count = 0
    
    for step in steps:
        PLAN.append({"step": step, "status": "pending"})
        added_count += 1
        
    return {
        "content": [{"type": "text", "text": f"Added {added_count} steps to plan."}]
    }


def read_plan(args: dict[str, Any]) -> dict[str, Any]:
    if not PLAN:
        return {"content": [{"type": "text", "text": "Plan is empty."}]}
    
    result = "Current Plan:\n"
    for i, item in enumerate(PLAN):
        icon = "✅" if item["status"] == "done" else "⬜"
        result += f"{i+1}. {icon} {item['step']}\n"
        
    return {
        "content": [{"type": "text", "text": result}]
    }


def mark_step_done(args: dict[str, Any]) -> dict[str, Any]:
    idx = args["index"] - 1
    if 0 <= idx < len(PLAN):
        PLAN[idx]["status"] = "done"
        return {
            "content": [{"type": "text", "text": f"Marked step #{args['index']} as complete."}]
        }
    else:
        return {
            "content": [{"type": "text", "text": f"Error: Invalid step index {args['index']}."}],
            "isError": True
        }


def main():
    # Call add_to_plan with a dictionary containing a "steps" key with a list of 2-3 step strings
    add_result = add_to_plan({"steps": ["Write code", "Test code", "Deploy code"]})
    
    # Print the result from add_to_plan
    print("--- 1. add_to_plan Result ---")
    print(add_result)
    print()
    
    # Call read_plan with an empty args dictionary {}
    read_result_1 = read_plan({})
    
    # Print the result from read_plan
    print("--- 2. read_plan Result (Initial) ---")
    print(read_result_1)
    print()
    
    # Call mark_step_done with a dictionary containing "index": 1
    mark_result = mark_step_done({"index": 1})
    
    # Print the result from mark_step_done
    print("--- 3. mark_step_done Result ---")
    print(mark_result)
    print()
    
    # Call read_plan again to see the updated state
    read_result_2 = read_plan({})
    
    # Print the final result
    print("--- 4. read_plan Result (Updated) ---")
    print(read_result_2)


if __name__ == "__main__":
    main()

```

### Architectural Insights Discovered:

* **The Ingestion Signature Pattern:** Notice how you had to invoke the functions by passing single dictionaries (like `{"steps": [...]}` or `{"index": 1}`). This simulates the precise JSON-RPC argument objects that Claude passes into your tools at runtime.
* **The Return Envelopes:** Even though the function calculations output straightforward logs, the return statements enforce the exact `"content"` payload mapping required by the SDK framework (${\text{\{"content": [\{"type": "text", "text": ...\}]\}}}$). This guarantees compatibility when these functions are hooked up to the agent ecosystem!

## Decorating Functions as MCP Tools

## Connecting Your First Custom Tools

## Configuring Agent for File Organization Task